# Clase 16: Seaborn y visualización estadística

Seaborn convierte datos en gráficos estadísticos con pocas líneas de código,
revelando patrones que las tablas ocultan.

**Datasets:** `ventas_tienda.csv` y `estudiantes.csv`

**Pregunta guía:** ¿Cómo elegir el gráfico correcto para responder cada pregunta sobre los datos?

In [ ]:
# Qué hace: importar las librerías necesarias y configurar el estilo visual
# Por qué sirve: establece un estilo uniforme para todos los gráficos de la sesión
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Estilo de gráficos: fondo blanco con grilla, paleta de colores suaves
sns.set_theme(style="whitegrid", palette="muted")

# Carga de datos
ventas = pd.read_csv("../../datasets/ventas_tienda.csv")
estudiantes = pd.read_csv("../../datasets/estudiantes.csv")

print("Ventas:", ventas.shape)
print(ventas.dtypes)
print()
print("Estudiantes:", estudiantes.shape)
print(estudiantes.dtypes)

In [ ]:
# Qué hace: crear un histograma con curva de densidad de las ventas diarias
# Por qué sirve: ver cómo se distribuyen los valores — si son simétricos, tienen cola, etc.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma: divide los valores en rangos y cuenta cuántos caen en cada uno
sns.histplot(data=ventas, x="total_venta", bins=30, kde=True,
             color="steelblue", ax=axes[0])
axes[0].set_title("Distribución de ventas diarias")
axes[0].set_xlabel("Total de venta")
axes[0].set_ylabel("Frecuencia")

# KDE puro: curva suave que estima la distribución sin barras
sns.kdeplot(data=ventas, x="total_venta", fill=True, color="coral", ax=axes[1])
axes[1].set_title("Densidad de ventas diarias")
axes[1].set_xlabel("Total de venta")

plt.tight_layout()
plt.show()

# Observación: si la curva tiene cola hacia la derecha, hay pocas ventas muy grandes

In [ ]:
# Qué hace: comparar la distribución de ventas entre categorías con un boxplot
# Por qué sirve: el boxplot muestra mediana, rango intercuartílico y valores atípicos

plt.figure(figsize=(11, 5))
sns.boxplot(data=ventas, x="categoria", y="total_venta",
            palette="Set2", width=0.5)
plt.title("Distribución de ventas por categoría de producto")
plt.xlabel("Categoría")
plt.ylabel("Total de venta")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Los puntos fuera de los bigotes son outliers: valores muy alejados del rango típico

In [ ]:
# Qué hace: comparar el violinplot con el boxplot para las notas por curso
# Por qué sirve: el violinplot agrega la forma de la distribución, no solo los percentiles

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot estándar
sns.boxplot(data=estudiantes, x="curso", y="nota_final",
            palette="pastel", ax=axes[0])
axes[0].set_title("Notas por curso — Boxplot")
axes[0].set_xlabel("Curso")
axes[0].set_ylabel("Nota final")

# Violinplot: muestra la densidad completa a cada lado
sns.violinplot(data=estudiantes, x="curso", y="nota_final",
               palette="pastel", inner="quartile", ax=axes[1])
axes[1].set_title("Notas por curso — Violinplot")
axes[1].set_xlabel("Curso")
axes[1].set_ylabel("Nota final")

plt.tight_layout()
plt.show()

In [ ]:
# Qué hace: mostrar el promedio de ventas por día de la semana con barplot
# Por qué sirve: las barras de error muestran cuánta variación hay en cada estimación

# Convertir fecha a datetime y extraer el nombre del día
ventas["fecha"] = pd.to_datetime(ventas["fecha"])
ventas["dia_semana"] = ventas["fecha"].dt.day_name(locale="es_ES.utf8") if False else ventas["fecha"].dt.dayofweek

orden_dias = [0, 1, 2, 3, 4, 5, 6]
nombres_dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

plt.figure(figsize=(9, 4))
sns.barplot(data=ventas, x="dia_semana", y="total_venta",
            palette="Blues_d", order=orden_dias, errorbar="ci")
plt.xticks(ticks=range(7), labels=nombres_dias)
plt.title("Promedio de ventas por día de la semana")
plt.xlabel("Día")
plt.ylabel("Venta promedio")
plt.tight_layout()
plt.show()

In [ ]:
# Qué hace: explorar la relación entre unidades vendidas y total de venta
# Por qué sirve: hue agrega una dimensión visual — podemos ver si las categorías se comportan diferente

plt.figure(figsize=(10, 5))
sns.scatterplot(data=ventas, x="unidades", y="total_venta",
                hue="categoria", size="descuento",
                sizes=(20, 200), alpha=0.65, palette="tab10")
plt.title("Unidades vendidas vs Total de venta por categoría")
plt.xlabel("Unidades vendidas")
plt.ylabel("Total de venta")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Categoría")
plt.tight_layout()
plt.show()

# Observación: si los puntos forman una línea diagonal, más unidades = más venta (obvio)
# La interesante es ver si alguna categoría rompe ese patrón

In [ ]:
# Qué hace: calcular y visualizar la matriz de correlación como heatmap
# Por qué sirve: permite ver de un vistazo qué variables están relacionadas entre sí

# Seleccionar solo variables numéricas
numericas_ventas = ventas.select_dtypes(include="number")

# Calcular correlaciones
correlacion = numericas_ventas.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(correlacion,
            annot=True,         # muestra los valores dentro de cada celda
            fmt=".2f",          # dos decimales
            cmap="coolwarm",    # rojo = correlación positiva, azul = negativa
            center=0,           # centra el colormap en 0
            square=True,
            linewidths=0.5)
plt.title("Matriz de correlación — Dataset de ventas")
plt.tight_layout()
plt.show()

In [ ]:
# Qué hace: explorar todas las relaciones entre variables numéricas de estudiantes con pairplot
# Por qué sirve: es la exploración más completa posible en un solo gráfico

# Seleccionar columnas relevantes para no sobrecargar el gráfico
cols_pairplot = ["nota_final", "asistencia", "horas_estudio", "modalidad"]
df_pair = estudiantes[cols_pairplot].dropna()

# pairplot con hue para colorear por modalidad
g = sns.pairplot(df_pair, hue="modalidad",
                 plot_kws={"alpha": 0.5},
                 diag_kind="kde")
g.figure.suptitle("Relaciones entre variables de estudiantes", y=1.02)
plt.show()

# La diagonal muestra la distribución de cada variable
# Los gráficos fuera de la diagonal muestran la relación entre cada par

In [ ]:
# Qué hace: personalizar el estilo visual de un gráfico completo
# Por qué sirve: aprender a ajustar paleta, tamaño y títulos para presentaciones

# Cambiar el tema completo
sns.set_theme(style="dark", palette="rocket")

plt.figure(figsize=(10, 5))
ax = sns.violinplot(data=estudiantes, x="curso", y="nota_final",
                    palette="rocket", inner="box")

# Personalización manual
ax.set_title("Distribución de notas por curso", fontsize=14, pad=12)
ax.set_xlabel("Curso", fontsize=11)
ax.set_ylabel("Nota final", fontsize=11)
ax.axhline(y=60, color="white", linestyle="--", linewidth=1.5, label="Mínimo aprobatorio")
ax.legend()

plt.tight_layout()
plt.show()

# Restaurar el tema para el resto del notebook
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
# Qué hace: crear un gráfico de dispersión de notas vs asistencia con línea de tendencia
# Por qué sirve: regplot combina scatterplot con una línea de regresión para ver la tendencia

plt.figure(figsize=(9, 5))
sns.regplot(data=estudiantes, x="asistencia", y="nota_final",
            scatter_kws={"alpha": 0.4, "color": "steelblue"},
            line_kws={"color": "red", "linewidth": 2})
plt.title("Relación entre asistencia y nota final")
plt.xlabel("Porcentaje de asistencia")
plt.ylabel("Nota final")
plt.tight_layout()
plt.show()

# La línea roja es la tendencia general; la zona sombreada es el intervalo de confianza
# Si la línea sube, mayor asistencia se asocia con mejor nota

## Resumen de la clase

| Gráfico | Cuándo usarlo | Pregunta que responde |
|---|---|---|
| `histplot` | Variable numérica continua | ¿Cómo se distribuyen los valores? |
| `kdeplot` | Variable numérica continua | ¿Cuál es la forma de la distribución? |
| `boxplot` | Numérica + categórica | ¿Cómo varía una variable entre grupos? |
| `violinplot` | Numérica + categórica | ¿Cuál es la forma completa de la distribución por grupo? |
| `barplot` | Numérica + categórica | ¿Cuál es el promedio por grupo? |
| `scatterplot` | Dos numéricas | ¿Están relacionadas dos variables? |
| `heatmap` | Matriz numérica | ¿Qué variables están correlacionadas entre sí? |
| `pairplot` | Varias numéricas | ¿Cuáles son todas las relaciones de un dataset? |

**Próxima clase:** Estadística inferencial — usaremos estas visualizaciones como punto de partida para preguntar si las diferencias que vemos son estadísticamente reales.